# TrOCR Fine-Tuning (Notebook)

Run cells top to bottom.

- This notebook uses the same training logic as `scripts/finetune_trocr.py`
- It is optimized for low VRAM GPUs (FP16 weights, gradient checkpointing, 8-bit Adam)
- Metrics shown each epoch: `train_loss` and `cer` (lower is better)

In [ ]:
from __future__ import annotations

import json
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from jiwer import cer
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

try:
    import bitsandbytes as bnb
    BNB_AVAILABLE = True
except ImportError:
    BNB_AVAILABLE = False


PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "scripts" else Path.cwd().resolve()
DATASET_ROOT = PROJECT_ROOT / "odia_dataset"
OUTPUT_DIR = PROJECT_ROOT / "saved_model"
MODEL_NAME = "microsoft/trocr-base-handwritten"


@dataclass
class TrainingConfig:
    model_name: str = MODEL_NAME
    dataset_root: Path = DATASET_ROOT
    output_dir: Path = OUTPUT_DIR
    batch_size: int = 1
    learning_rate: float = 5e-5
    epochs: int = 1
    max_target_length: int = 128
    num_beams: int = 1
    num_workers: int = 2
    seed: int = 42
    device: str = "cuda"
    use_mixed_precision: bool = True
    use_8bit_adam: bool = True
    grad_accum_steps: int = 8
    eval_samples: int = 500

print("Cell 2 ready: imports + config loaded.")

/home/mors/Code/odia_Model/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cell 2 ready: imports + config loaded.


In [ ]:
class OdiaOCRDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, dataset_root: Path) -> None:
        self.dataframe = dataframe.reset_index(drop=True)
        self.dataset_root = dataset_root

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.dataframe.iloc[index]
        image_path = self.dataset_root / str(row["image_name"])
        text = str(row["text"])
        image = Image.open(image_path).convert("RGB")
        return {
            "image": image,
            "text": text,
            "image_path": str(image_path),
        }


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_split(dataset_root: Path, split_name: str) -> pd.DataFrame:
    csv_path = dataset_root / f"{split_name}_labels.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing labels file: {csv_path}")
    dataframe = pd.read_csv(csv_path)
    required_columns = {"image_name", "text"}
    missing_columns = required_columns.difference(dataframe.columns)
    if missing_columns:
        raise ValueError(f"{csv_path.name} is missing required columns: {sorted(missing_columns)}")
    return dataframe


def build_collate_fn(processor: TrOCRProcessor, max_target_length: int):
    def collate_fn(batch: list[dict[str, Any]]) -> dict[str, Any]:
        images = [item["image"] for item in batch]
        texts = [item["text"] for item in batch]
        pixel_values = processor(images=images, return_tensors="pt").pixel_values
        tokenized = processor.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_target_length,
            return_tensors="pt",
        )
        labels = tokenized.input_ids.clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        return {
            "pixel_values": pixel_values,
            "labels": labels,
            "texts": texts,
            "image_paths": [item["image_path"] for item in batch],
        }

    return collate_fn


print("Cell 3 ready: dataset + loaders helpers loaded.")

Cell 3 ready: dataset + loaders helpers loaded.


In [ ]:
def evaluate(
    model: VisionEncoderDecoderModel,
    processor: TrOCRProcessor,
    dataloader: DataLoader,
    device: torch.device,
    max_target_length: int,
    num_beams: int,
    use_mixed_precision: bool,
) -> float:
    model.eval()
    predictions: list[str] = []
    references: list[str] = []
    use_amp = use_mixed_precision and device.type == "cuda"

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False, unit="batch"):
            pixel_values = batch["pixel_values"].to(device)
            if use_amp:
                pixel_values = pixel_values.half()
            generated_ids = model.generate(
                pixel_values,
                max_new_tokens=max_target_length,
                num_beams=num_beams,
            )
            decoded_predictions = processor.batch_decode(generated_ids, skip_special_tokens=True)
            predictions.extend(prediction.strip() for prediction in decoded_predictions)
            references.extend(reference.strip() for reference in batch["texts"])

    return cer(references, predictions)


def save_metadata(output_dir: Path, config: TrainingConfig, best_cer: float, best_epoch: int) -> None:
    metadata = {
        "config": asdict(config),
        "best_cer": best_cer,
        "best_epoch": best_epoch,
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    (output_dir / "training_metadata.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")


print("Cell 4 ready: evaluate + metadata helpers loaded.")

Cell 4 ready: evaluate + metadata helpers loaded.


In [ ]:
def run_training(config: TrainingConfig) -> None:
    set_seed(config.seed)

    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    if config.device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but unavailable. Set device='cpu' in config.")

    device = torch.device(config.device)
    use_amp = config.use_mixed_precision and device.type == "cuda"

    train_df = load_split(config.dataset_root, "train")
    test_df = load_split(config.dataset_root, "test")

    processor = TrOCRProcessor.from_pretrained(config.model_name, use_fast=False)
    model = VisionEncoderDecoderModel.from_pretrained(config.model_name)
    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.eos_token_id = processor.tokenizer.sep_token_id
    model.generation_config.max_new_tokens = config.max_target_length
    model.to(device)

    if use_amp:
        model.half()
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        print("Model weights converted to FP16 and gradient checkpointing enabled.")

    train_dataset = OdiaOCRDataset(train_df, config.dataset_root)
    eval_df = test_df if config.eval_samples < 0 else test_df.iloc[: config.eval_samples]
    test_dataset = OdiaOCRDataset(eval_df, config.dataset_root)
    collate_fn = build_collate_fn(processor, config.max_target_length)

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=device.type == "cuda",
        collate_fn=collate_fn,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=device.type == "cuda",
        collate_fn=collate_fn,
    )
    print(f"Evaluating on {len(eval_df)} test samples per epoch (use eval_samples=-1 for all 2000).")

    if config.use_8bit_adam and BNB_AVAILABLE and device.type == "cuda":
        optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=config.learning_rate)
        print("Using 8-bit AdamW (bitsandbytes) - optimizer state VRAM reduced ~75%.")
    else:
        from torch.optim import AdamW

        optimizer = AdamW(model.parameters(), lr=config.learning_rate)
        if config.use_8bit_adam and not BNB_AVAILABLE:
            print("Warning: bitsandbytes unavailable, falling back to standard AdamW.")

    best_cer = float("inf")
    best_epoch = 0
    config.output_dir.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, config.epochs + 1):
        model.train()
        running_loss = 0.0
        accum_steps = max(1, config.grad_accum_steps)
        optimizer.zero_grad(set_to_none=True)

        for step_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{config.epochs}", unit="batch")):
            pixel_values = batch["pixel_values"].to(device)
            if use_amp:
                pixel_values = pixel_values.half()
            labels = batch["labels"].to(device)

            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss / accum_steps
            loss.backward()

            if (step_idx + 1) % accum_steps == 0 or (step_idx + 1) == len(train_loader):
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            running_loss += loss.item() * accum_steps

        average_loss = running_loss / max(1, len(train_loader))
        cer_score = evaluate(
            model=model,
            processor=processor,
            dataloader=test_loader,
            device=device,
            max_target_length=config.max_target_length,
            num_beams=config.num_beams,
            use_mixed_precision=config.use_mixed_precision,
        )
        char_acc = max(0.0, 1.0 - cer_score) * 100.0
        print(f"Epoch {epoch}: train_loss={average_loss:.4f} cer={cer_score:.4f} char_acc={char_acc:.2f}%")

        if cer_score < best_cer:
            best_cer = cer_score
            best_epoch = epoch
            model.save_pretrained(config.output_dir)
            processor.save_pretrained(config.output_dir)
            save_metadata(config.output_dir, config, best_cer, best_epoch)

    print(f"Best model saved to {config.output_dir} with CER={best_cer:.4f} at epoch {best_epoch}")


print("Cell 5 ready: training loop loaded.")

Cell 5 ready: training loop loaded.


In [ ]:
# Run after executing Cells 2, 3, 4, and 5.
# Quick smoke test:
cfg = TrainingConfig(
    batch_size=1,
    epochs=10,
    eval_samples=-1,   # full test set (all 2000 test images)
    num_beams=1,
    grad_accum_steps=8,
    use_mixed_precision=True,
    use_8bit_adam=True,
)

run_training(cfg)

# Full run example:
# cfg.epochs = 10
# cfg.eval_samples = 500
# run_training(cfg)

SyntaxError: incomplete input (1823204197.py, line 6)